# AdjointRWM production pilot v2: fixed allocation contract and rival allocators

Pilot v2 (roadmap **E2.1 + E2.2**, plan milestone **B2**) retrains the pilot's DROID-100 world model and then asks the H2 question on it: does the amortised **co-state** allocator choose refinements better than a matched **direct critic** and every other required allocator? v1 (`AdjointRWM_Production_Pilot.ipynb`) is kept unchanged for provenance. Its findings are in `docs/research-notes/2026-09-29-droid100-pilot-findings.md`, and every change below fixes a problem listed there in §2.4.

| v1 problem | v2 contract |
|---|---|
| No hold option | Option 0 is `hold` (zero effect, zero cost, exact gain 0) |
| Candidates trained in random subsets, scored alone | Stage 1 trains single-choice refinement (hold or one candidate), as scored |
| "Random" was one draw | Exact expected-random, `always_hold`, `always_c1..c4` |
| Traces lacked gains | Full `exact_gain[N, 5]` in every trace |
| Gate label almost never positive | Sign of `critic_regret − adjoint_regret`, ties weighted 0, class weight from validation |
| Co-state and critic in one loss, critic head ~20 % smaller | Separate jobs, same batches/steps/optimiser, parameter-matched heads |
| One seed | 5 paired seeds on the frozen pilot split |
| Dynamics gate read the test split | Dynamics gate on validation; test is read once, at the end |

**Allocator arms:** `adjoint` (amortised co-state, first-order score), `critic` (direct marginal-gain critic), `adjoint_randomized` (co-state permuted across the batch: control), `uncertainty` (reduction in the model's own expected NLL), `hybrid` (gate), fixed and random baselines, and two **non-deployable** diagnostics: `exact_costate` (autograd ∂J/∂z, a ceiling) and `oracle` (defines regret).

**Gates** (plan §5): (1) opportunity on validation, (2) critic realisation floor on validation, (3) primary endpoint `adjoint − critic` test regret with an episode-cluster bootstrap CI, per seed and pooled, (4) randomized control. If gate 1 or 2 fails, H2 on this benchmark is **non-diagnostic** or **under-realised**, and no adjoint claim follows whatever the regrets say. The co-state here is with respect to the latent only (findings §2.4.5), so report it as a latent-state co-state.

**How to run:** Colab GPU runtime (L4), Drive mounted, run all. To resume a paused run, set `RESUME_RUN_ID` in section 1, reconnect to the same GPU type and run all again.

In [ ]:
# ================================================================
# 0. Install, fetch the tested package, and pin its commit
# ================================================================
%pip -q install --upgrade "tensorflow-datasets>=4.9" rlds pyarrow

import os
import subprocess
import sys

REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA. Confirmatory runs pin a commit SHA.
REPO_DIR = '/content/para_001'

if not os.path.exists(REPO_DIR):
    # Private repository: authenticate first (for example a GitHub token from Colab secrets),
    # or upload a checkout of the repository to REPO_DIR.
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)


def resolve_ref(ref):
    for candidate in (f'origin/{ref}', ref):
        result = subprocess.run(
            ['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'Cannot resolve {ref!r} in {REPO_URL}.')


subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolve_ref(REPO_REF)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

try:
    import adjointrwm
    import adjointrwm.allocators
    import adjointrwm.models
    import adjointrwm.training  # noqa: F401
except ImportError as error:
    raise RuntimeError(
        f'{REPO_REF!r} ({REPO_COMMIT[:12]}) predates the v2 modules. '
        'Set REPO_REF to the branch or commit that contains src/adjointrwm/allocators.py.'
    ) from error
print('adjointrwm', adjointrwm.__version__, 'at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '')

In [ ]:
# ================================================================
# 1. Runtime, frozen configuration, immutable run directory
# ================================================================
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

import datetime
import json
import platform
import traceback
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from adjointrwm.io import append_jsonl, atomic_copy, atomic_write_json, atomic_write_text, sha256_json

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if not any(name in GPU_NAME for name in ('L4', 'A100', 'H100')):
    raise RuntimeError(f'Unsupported hardware profile: {GPU_NAME}')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


@dataclass(frozen=True)
class PilotV2Config:
    # Data: identical to pilot v1.
    dataset_name: str = 'droid_100'
    dataset_data_dir: str = 'gs://gresearch/robotics'
    max_episodes: int = 100
    frame_stride: int = 2
    context_len: int = 8
    horizon: int = 4
    window_stride: int = 2
    pilot_run_id: str = 'droid100_adjoint_20260929T070629Z'
    image_size: int = 128
    embedding_batch_size: int = 64

    # Model: pilot v1 values; the candidate regime is the v2 contract.
    d_model: int = 512
    transformer_layers: int = 6
    transformer_heads: int = 8
    transformer_ff: int = 2048
    dropout: float = 0.10
    num_refinement_candidates: int = 4
    candidate_costs: tuple = (1.0, 1.0, 1.5, 2.0)
    mask_mode: str = 'single_choice'   # 'single_choice_wta' is a declared variant, not the default
    prediction_mode: str = 'base'      # hold is the default action in v2
    rate_beta: float = 0.002
    candidate_cost_weight: float = 0.002

    # Training (pilot v1 values unless noted).
    seeds: tuple = (0, 1, 2, 3, 4)
    seed_base: int = 20260928
    dynamics_steps: int = 1500
    allocator_steps: int = 750
    gate_steps: int = 300               # new in v2 (separate gate job)
    batch_size: int = 64
    learning_rate: float = 3.0e-4
    allocator_learning_rate: float = 1.0e-4
    weight_decay: float = 0.05
    warmup_steps: int = 100
    gradient_clip: float = 1.0
    eval_interval: int = 100
    checkpoint_interval: int = 250
    eval_batch_size: int = 128

    # Gates.
    baseline_improvement_margin: float = 0.02
    opportunity_min_relative_headroom: float = 0.15
    bootstrap_resamples: int = 10000


CFG = PilotV2Config()
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)

RESUME_RUN_ID = None  # e.g. 'droid100_adjoint_v2_20261001T090000Z'
MAX_STEPS_PER_SESSION = None  # optimiser steps this session may run (None = no limit); not part of the frozen config

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production')
SESSION_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    stored = json.loads((RUN_DIR / 'config' / 'run_config.json').read_text())
    if stored['config_sha256'] != CONFIG_HASH:
        raise RuntimeError('This notebook config differs from the paused run. Resume with the identical config.')
    if stored['repo_commit'] != REPO_COMMIT:
        raise RuntimeError(f"The run used package commit {stored['repo_commit']}; set REPO_REF to it.")
    if stored['gpu'] != GPU_NAME:
        raise RuntimeError(f"The run started on {stored['gpu']}; reconnect to that GPU type.")
else:
    RUN_ID = 'droid100_adjoint_v2_' + SESSION_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    if RUN_DIR.exists():
        raise RuntimeError(f'{RUN_DIR} already exists; run directories are immutable.')

LOCAL = Path('/content/adjoint_rwm_work') / RUN_ID
PATHS = {name: RUN_DIR / name for name in ('config', 'cache', 'jobs', 'artifacts', 'benchmarks', 'figures', 'logs', 'reports', 'tests')}
for path in [LOCAL, *PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

SOURCE_MANIFEST = {
    'dataset': CFG.dataset_name, 'data_dir': CFG.dataset_data_dir, 'format': 'RLDS / TensorFlow Datasets',
    'licence': 'DROID: CC BY 4.0; attribute the DROID authors', 'synthetic_fallback_allowed': False,
    'repo_url': REPO_URL, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY,
}
SOURCE_HASH = sha256_json(SOURCE_MANIFEST)
if not RESUME_RUN_ID:
    atomic_write_json(PATHS['config'] / 'run_config.json', {
        'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT,
        'gpu': GPU_NAME, 'notebook': 'notebooks/01-production/AdjointRWM_Production_Pilot_v2.ipynb',
    })
    atomic_write_json(PATHS['config'] / 'source_manifest.json', {**SOURCE_MANIFEST, 'source_sha256': SOURCE_HASH})
atomic_write_json(PATHS['config'] / f'environment_{SESSION_ID}.json', {
    'run_id': RUN_ID, 'session': SESSION_ID, 'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': GPU_NAME, 'gpu_vram_gib': GPU_VRAM_GIB,
    'precision': 'BF16 autocast for training; FP32 exact co-state and gain labels',
})
atomic_write_text(PATHS['config'] / f'pip_freeze_{SESSION_ID}.txt',
                  subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

METRICS_LOCAL = LOCAL / f'metrics_{SESSION_ID}.jsonl'


def log(record):
    append_jsonl(METRICS_LOCAL, {'time_utc': datetime.datetime.now(datetime.UTC).isoformat(), **record})


def sync_logs():
    if METRICS_LOCAL.exists():
        atomic_copy(METRICS_LOCAL, PATHS['logs'] / METRICS_LOCAL.name)


print('Run:', RUN_ID, '| resumed' if RESUME_RUN_ID else '| new', '| GPU:', GPU_NAME)
print('Run directory:', RUN_DIR)

In [ ]:
# ================================================================
# 2. Real DROID episodes -> frozen ResNet-18 features (cached once per run), split parity
# ================================================================
import matplotlib.pyplot as plt

from adjointrwm.data import (
    SPLITS, assignment_from_manifest, build_feature_cache, compare_splits, episode_split, load_droid,
    persist_cache, restore_cache, split_sets, state_groups, write_cache_manifest,
)
from adjointrwm.features import ResNet18Embedder

CACHE_MANIFEST = PATHS['cache'] / 'cache_manifest.json'
LOCAL_CACHE = LOCAL / 'cache'

if CACHE_MANIFEST.exists():
    cache_manifest = json.loads(CACHE_MANIFEST.read_text())
    records = restore_cache(cache_manifest, LOCAL_CACHE)
    print(f'Restored {len(records)} cached episodes (hashes verified).')
else:
    import tensorflow_datasets as tfds

    raw_dataset, dataset_info = load_droid(CFG.dataset_name, CFG.dataset_data_dir)
    atomic_write_text(PATHS['config'] / 'tfds_builder_info.txt', str(dataset_info))
    embedders = {'resnet18': ResNet18Embedder(DEVICE, image_size=CFG.image_size, batch_size=CFG.embedding_batch_size)}
    inspection = []

    def keep_for_inspection(episode_id, parsed):
        if len(inspection) < 8:
            inspection.append((episode_id, parsed['images']['exterior_image_1_left'][0],
                               parsed['images']['wrist_image_left'][0], parsed['states'], parsed['actions']))

    def droid_episodes():
        for index, episode in enumerate(raw_dataset.take(CFG.max_episodes)):
            yield index, episode, list(tfds.as_numpy(episode['steps']))

    records, skipped = build_feature_cache(
        droid_episodes(), embedders, LOCAL_CACHE, frame_stride=CFG.frame_stride,
        min_steps=CFG.context_len + CFG.horizon + 1, on_episode=keep_for_inspection,
    )
    if len(records) < 20:
        raise RuntimeError(f'Only {len(records)} valid episodes were cached; at least 20 are required.')
    records = persist_cache(records, PATHS['cache'] / 'episodes')
    cache_manifest = write_cache_manifest(records, skipped, CACHE_MANIFEST, {
        'encoders': {name: e.manifest() for name, e in embedders.items()}, 'source_sha256': SOURCE_HASH,
    })
    fig, axes = plt.subplots(len(inspection), 3, figsize=(14, 3 * len(inspection)))
    for row, (episode_id, exterior, wrist, states, actions) in enumerate(inspection):
        axes[row, 0].imshow(exterior); axes[row, 0].set_title(f'{episode_id} exterior'); axes[row, 0].axis('off')
        axes[row, 1].imshow(wrist); axes[row, 1].set_title('wrist'); axes[row, 1].axis('off')
        axes[row, 2].plot(states[:, 0], label='state[0]'); axes[row, 2].plot(actions[:, 0], label='action[0]')
        axes[row, 2].legend(); axes[row, 2].set_title('trajectory channels')
    plt.tight_layout()
    fig.savefig(PATHS['figures'] / 'episode_inspection.png', dpi=120)
    plt.show()
    plt.close(fig)
    del embedders
    torch.cuda.empty_cache()

assignment = episode_split([r['episode_id'] for r in records])
for record in records:
    record['split'] = assignment[record['episode_id']]
PILOT_MANIFEST = DRIVE_ROOT / 'runs' / CFG.pilot_run_id / 'config' / 'data_manifest.json'
if PILOT_MANIFEST.exists():
    SPLIT_PARITY = compare_splits(assignment, assignment_from_manifest(json.loads(PILOT_MANIFEST.read_text())))
    if not SPLIT_PARITY['matches']:
        raise RuntimeError(f'The episode split differs from pilot v1: {SPLIT_PARITY}')
else:
    SPLIT_PARITY = {'matches': None, 'reason': f'{PILOT_MANIFEST} not found; same rule, not compared'}
sets = split_sets(assignment)
assert sets['train'].isdisjoint(sets['validation']) and sets['train'].isdisjoint(sets['test'])
assert sets['validation'].isdisjoint(sets['test'])

GROUPS = state_groups(records[0]['state_keys'], records[0]['state_dims'])
STABLE_FIELDS = ('episode_index', 'episode_id', 'cached_sha256', 'length', 'split', 'instruction')
DATA_MANIFEST = {
    'source_sha256': SOURCE_HASH, 'frame_stride': CFG.frame_stride, 'state_keys': records[0]['state_keys'],
    'state_dims': records[0]['state_dims'], 'action_dim': records[0]['action_dim'], 'encoders': cache_manifest['encoders'],
    'episodes': [{k: r[k] for k in STABLE_FIELDS} for r in sorted(records, key=lambda r: r['episode_id'])],
    'skipped': cache_manifest['skipped'], 'split_counts': {s: len(v) for s, v in sets.items()},
}
DATA_MANIFEST_HASH = sha256_json(DATA_MANIFEST)
atomic_write_json(PATHS['config'] / 'data_manifest.json', {**DATA_MANIFEST, 'data_manifest_sha256': DATA_MANIFEST_HASH})
DATA_AUDIT = {
    'valid_episode_count': len(records), 'split_counts': DATA_MANIFEST['split_counts'], 'split_parity_with_pilot': SPLIT_PARITY,
    'real_image_payloads_verified': True, 'split_disjointness_verified': True, 'synthetic_fallback_used': False, 'status': 'PASS',
}
atomic_write_json(PATHS['tests'] / 'data_audit.json', DATA_AUDIT)
print(json.dumps(DATA_AUDIT, indent=2))

In [ ]:
# ================================================================
# 3. Windows (pilot layout), normalisation, model factory
# ================================================================
from adjointrwm.data import WindowDataset, WindowSpec, fit_normaliser
from adjointrwm.models import AdjointRecursiveWorldModel, AdjointRWMConfig

SPEC = WindowSpec(CFG.context_len, CFG.horizon, CFG.window_stride)
BY_SPLIT = {s: [r for r in records if r['split'] == s] for s in SPLITS}


def load_arrays(record, keys=('states', 'actions')):
    with np.load(record['cached_path']) as episode:
        return {k: episode[k] for k in keys}


train_arrays = [load_arrays(r) for r in BY_SPLIT['train']]
NORMALISATION = fit_normaliser([a['states'] for a in train_arrays], [a['actions'] for a in train_arrays])
np.savez(PATHS['config'] / 'normalisation.npz', **NORMALISATION)
DATASETS = {s: WindowDataset(BY_SPLIT[s], SPEC, NORMALISATION, visual_layout='flat') for s in SPLITS}
print('Windows:', {s: len(d) for s, d in DATASETS.items()})

MODEL_CONFIG = AdjointRWMConfig(
    context_len=CFG.context_len, horizon=CFG.horizon, d_model=CFG.d_model, transformer_layers=CFG.transformer_layers,
    transformer_heads=CFG.transformer_heads, transformer_ff=CFG.transformer_ff, dropout=CFG.dropout,
    num_refinement_candidates=CFG.num_refinement_candidates, candidate_costs=CFG.candidate_costs,
    rate_beta=CFG.rate_beta, mask_mode=CFG.mask_mode, prediction_mode=CFG.prediction_mode,
)
sample = DATASETS['train'][0]
STATE_DIM, ACTION_DIM, VISUAL_DIM = sample['context_state'].shape[-1], sample['context_action'].shape[-1], sample['context_visual'].shape[-1]


def build_dynamics_model():
    return AdjointRecursiveWorldModel(STATE_DIM, ACTION_DIM, VISUAL_DIM, MODEL_CONFIG)


reference = build_dynamics_model()
MODEL_MANIFEST = {
    'class': 'AdjointRecursiveWorldModel', 'config': MODEL_CONFIG.to_dict(),
    'trainable_parameters': sum(p.numel() for p in reference.parameters() if p.requires_grad),
    'prediction_parameters': reference.prediction_parameters(),
    'state_dim': STATE_DIM, 'action_dim': ACTION_DIM, 'visual_dim': VISUAL_DIM,
    'visual_target': 'frozen ResNet-18 ImageNet weights',
    'note': 'v2 trains the allocator heads as separate modules (adjointrwm.allocators); the in-model heads stay unused.',
}
atomic_write_json(PATHS['config'] / 'model_manifest.json', MODEL_MANIFEST)
del reference
print(json.dumps(MODEL_MANIFEST, indent=2))

In [ ]:
# ================================================================
# 4. Stage 1 per seed: dynamics (single-choice candidates) + dynamics gate on validation
# ================================================================
from adjointrwm.eval import horizon_mean_rmse, per_window_mse, persistence_forecast, stack_windows
from adjointrwm.training import TrainConfig, load_best_weights, predict_dataset, seed_everything, train_job

SESSION = {'steps_left': MAX_STEPS_PER_SESSION}


def identity(stage, seed, extra=''):
    return {'run_id': RUN_ID, 'arm': stage, 'seed': int(seed), 'config_hash': sha256_json([CONFIG_HASH, extra]),
            'data_manifest_hash': DATA_MANIFEST_HASH, 'source_hash': SOURCE_HASH}


def job_dir(seed, stage):
    return PATHS['jobs'] / f'seed_{seed}' / stage


def train_config(steps, lr, seed, warmup):
    return TrainConfig(steps=steps, batch_size=CFG.batch_size, lr=lr, weight_decay=CFG.weight_decay, warmup_steps=warmup,
                       grad_clip=CFG.gradient_clip, eval_interval=CFG.eval_interval, checkpoint_interval=CFG.checkpoint_interval,
                       seed=CFG.seed_base + seed, amp=True, eval_batch_size=CFG.eval_batch_size)


def run(stage, seed, model, cfg, extra='', **kwargs):
    persist = job_dir(seed, stage)
    for marker in ('FAILED.json', 'DONE.json'):
        if (persist / marker).exists():
            return json.loads((persist / marker).read_text())
    if SESSION['steps_left'] is not None and SESSION['steps_left'] <= 0:
        return {'status': 'PAUSED', 'stage': stage, 'seed': seed}
    try:
        summary = train_job(model, DATASETS['train'], DATASETS['validation'], cfg, identity=identity(stage, seed, extra),
                            local_dir=LOCAL / 'jobs' / f'seed_{seed}' / stage, persist_dir=persist, device=DEVICE,
                            max_steps_this_session=SESSION['steps_left'], stage=stage,
                            log=lambda r: log({'job': f'seed_{seed}/{stage}', **r}), **kwargs)
    except FloatingPointError as error:
        summary = {**identity(stage, seed, extra), 'status': 'FAILED', 'error': repr(error), 'traceback': traceback.format_exc()}
        atomic_write_json(persist / 'FAILED.json', summary)
    if SESSION['steps_left'] is not None:
        SESSION['steps_left'] -= summary.get('session_steps', 0)
    sync_logs()
    print(f"seed {seed} {stage:>18}: {summary['status']}", summary.get('best', {}).get('validation_score', ''))
    return summary


def require_terminal(summaries, stage):
    pending = [k for k, v in summaries.items() if v['status'] not in ('DONE', 'FAILED')]
    if pending:
        raise RuntimeError(f'{stage}: {len(pending)} job(s) PAUSED. Set RESUME_RUN_ID = {RUN_ID!r} and run all cells again.')


def load_dynamics(seed):
    model = build_dynamics_model().to(DEVICE)
    load_best_weights(model, job_dir(seed, 'dynamics'), identity=identity('dynamics', seed))
    return model.eval()


DYNAMICS = {}
for seed in CFG.seeds:
    seed_everything(CFG.seed_base + seed)
    DYNAMICS[seed] = run('dynamics', seed, build_dynamics_model(), train_config(CFG.dynamics_steps, CFG.learning_rate, seed, CFG.warmup_steps))
require_terminal(DYNAMICS, 'Dynamics')

VAL = stack_windows(DATASETS['validation'], keys=['context_state', 'target_state'])
persistence_val = horizon_mean_rmse(per_window_mse(persistence_forecast(VAL['context_state'], CFG.horizon), VAL['target_state']))
DYNAMICS_GATE = {}
for seed, summary in DYNAMICS.items():
    if summary['status'] != 'DONE':
        DYNAMICS_GATE[seed] = {'passed': False, 'reason': 'dynamics job failed'}
        continue
    pred = predict_dataset(load_dynamics(seed), DATASETS['validation'], CFG.eval_batch_size, DEVICE)
    model_rmse = horizon_mean_rmse(per_window_mse(pred['state_mean'], pred['target_state']))
    improvement = (persistence_val - model_rmse) / persistence_val
    DYNAMICS_GATE[seed] = {'split': 'validation', 'prediction_mode': CFG.prediction_mode, 'model_rmse': model_rmse,
                           'persistence_rmse': persistence_val, 'relative_improvement': improvement,
                           'required_margin': CFG.baseline_improvement_margin,
                           'passed': bool(improvement >= CFG.baseline_improvement_margin)}
    torch.cuda.empty_cache()
atomic_write_json(PATHS['artifacts'] / 'dynamics_gate.json', DYNAMICS_GATE)
print(pd.DataFrame(DYNAMICS_GATE).T.to_string())

In [ ]:
# ================================================================
# 5. Stage 2 per seed: co-state, critic (separate, matched) and gate jobs on the frozen model
# ================================================================
from adjointrwm.allocators import AllocatorJob, allocation_traces, gate_positive_weight, validation_regret
from adjointrwm.io import sha256_file


def allocator_job(seed, head, teacher, pos_weight=1.0):
    seed_everything(CFG.seed_base + seed)
    job = AllocatorJob(teacher, head, CFG.candidate_cost_weight, gate_pos_weight=pos_weight)
    if head == 'gate':
        for trained in ('costate', 'critic'):
            job_trained = AllocatorJob(teacher, trained, CFG.candidate_cost_weight)
            load_best_weights(job_trained, job_dir(seed, f'allocator_{trained}'), identity=identity(f'allocator_{trained}', seed, TEACHER_HASH[seed]))
            getattr(job, trained).load_state_dict(getattr(job_trained, trained).state_dict())
            for p in getattr(job, trained).parameters():
                p.requires_grad_(False)
    return job.to(DEVICE)


def score(policy):
    return lambda job: validation_regret(job, DATASETS['validation'], CFG.eval_batch_size, DEVICE, policy)


TEACHER_HASH, ALLOCATOR, GATE_WEIGHTS = {}, {}, {}
for seed in CFG.seeds:
    if not DYNAMICS_GATE[seed]['passed']:
        print(f'seed {seed}: dynamics gate failed -> allocator training blocked for this seed (reported, not replaced)')
        continue
    TEACHER_HASH[seed] = sha256_file(job_dir(seed, 'dynamics') / 'best.pt')
    teacher = load_dynamics(seed)
    for head, policy in (('costate', 'adjoint'), ('critic', 'critic')):
        ALLOCATOR[(seed, head)] = run(
            f'allocator_{head}', seed, allocator_job(seed, head, teacher),
            train_config(CFG.allocator_steps, CFG.allocator_learning_rate, seed, warmup=0),
            extra=TEACHER_HASH[seed], score_fn=score(policy), score_name=f'validation_regret_{policy}',
        )
    if all(ALLOCATOR[(seed, h)]['status'] == 'DONE' for h in ('costate', 'critic')):
        probe = allocator_job(seed, 'gate', teacher)
        GATE_WEIGHTS[seed] = gate_positive_weight(allocation_traces(probe, DATASETS['validation'], CFG.eval_batch_size, DEVICE))
        ALLOCATOR[(seed, 'gate')] = run(
            'allocator_gate', seed, allocator_job(seed, 'gate', teacher, GATE_WEIGHTS[seed]['pos_weight']),
            train_config(CFG.gate_steps, CFG.allocator_learning_rate, seed, warmup=0),
            extra=TEACHER_HASH[seed], score_fn=score('hybrid'), score_name='validation_regret_hybrid',
        )
    del teacher
    torch.cuda.empty_cache()
require_terminal(ALLOCATOR, 'Allocators')
atomic_write_json(PATHS['artifacts'] / 'gate_class_balance.json', GATE_WEIGHTS)

In [ ]:
# ================================================================
# 6. Traces on validation and test (test read once), gates, H2 endpoint
# ================================================================
from adjointrwm.analysis import critic_realization_floor, episode_bootstrap_ci, policy_regret_frame, summarize_policies

WEIGHTED_COSTS = [0.0] + [c * CFG.candidate_cost_weight for c in CFG.candidate_costs]
TRACES, SUMMARIES, SEED_RESULTS = [], {}, {}
for seed in CFG.seeds:
    if ALLOCATOR.get((seed, 'gate'), {}).get('status') != 'DONE':
        SEED_RESULTS[seed] = {'status': 'NOT_EVALUATED', 'reason': 'an upstream job failed or its gate blocked it'}
        continue
    teacher = load_dynamics(seed)
    job = allocator_job(seed, 'gate', teacher)
    load_best_weights(job, job_dir(seed, 'allocator_gate'), identity=identity('allocator_gate', seed, TEACHER_HASH[seed]))
    for split in ('validation', 'test'):
        traces = allocation_traces(job, DATASETS[split], CFG.eval_batch_size, DEVICE)
        traces.insert(0, 'seed', seed)
        traces.insert(1, 'split', split)
        TRACES.append(traces)
        SUMMARIES[(seed, split)] = summarize_policies(traces, WEIGHTED_COSTS, num_resamples=CFG.bootstrap_resamples)
    val, test = SUMMARIES[(seed, 'validation')], SUMMARIES[(seed, 'test')]
    opp = val['opportunity']
    opportunity_ok = bool(opp['headroom_over_random'] > 0
                          and opp['headroom_over_best_fixed'] >= CFG.opportunity_min_relative_headroom * opp['headroom_over_random'])
    floor = critic_realization_floor(val)
    primary = test['paired_differences']['adjoint-critic']
    control = test['paired_differences']['adjoint_randomized-adjoint']
    if not opportunity_ok:
        verdict = 'NON_DIAGNOSTIC'
    elif not floor['passed']:
        verdict = 'UNDER_REALIZED'
    elif primary['ci_high'] < 0:
        verdict = 'ADJOINT_LOWER_REGRET'
    elif primary['ci_low'] > 0:
        verdict = 'CRITIC_LOWER_REGRET'
    else:
        verdict = 'TIE_OR_INCONCLUSIVE'
    SEED_RESULTS[seed] = {'status': 'EVALUATED', 'opportunity_gate_validation': opportunity_ok, 'critic_floor_validation': floor,
                          'adjoint_minus_critic_test': primary, 'randomized_minus_adjoint_test': control, 'verdict': verdict}
    del teacher, job
    torch.cuda.empty_cache()

ALL_TRACES = pd.concat(TRACES, ignore_index=True)
ALL_TRACES.to_parquet(PATHS['artifacts'] / 'allocation_traces_v2.parquet', index=False)
atomic_write_json(PATHS['artifacts'] / 'allocator_summaries.json', {f'seed_{s}/{split}': v for (s, split), v in SUMMARIES.items()})

# Pooled over seeds: regrets of every evaluated seed, clustered by test episode.
test_traces = ALL_TRACES[ALL_TRACES['split'] == 'test']
POOLED = {}
if len(test_traces):
    regrets = policy_regret_frame(test_traces)
    for a, b in (('adjoint', 'critic'), ('adjoint_randomized', 'adjoint'), ('hybrid', 'critic'), ('critic', 'uncertainty'), ('exact_costate', 'adjoint')):
        POOLED[f'{a}-{b}'] = episode_bootstrap_ci(regrets, a, baseline=b, num_resamples=CFG.bootstrap_resamples)
    POOLED['mean_regret'] = {c: float(regrets[c].mean()) for c in regrets.columns if c != 'episode_id'}
print(pd.DataFrame({s: {'verdict': r.get('verdict', r['status']),
                        'adjoint-critic': r.get('adjoint_minus_critic_test', {}).get('estimate'),
                        'CI': (r.get('adjoint_minus_critic_test', {}).get('ci_low'), r.get('adjoint_minus_critic_test', {}).get('ci_high'))}
                    for s, r in SEED_RESULTS.items()}).T.to_string())
print(json.dumps(POOLED, indent=1))

In [ ]:
# ================================================================
# 7. Test dynamics, contract tests, systems cost, report
# ================================================================
from adjointrwm.allocators import exact_targets
from adjointrwm.models import inputs_only
from adjointrwm.training import measure_latency_ms, move_batch

TEST = stack_windows(DATASETS['test'], keys=['context_state', 'target_state'])
persistence_test = horizon_mean_rmse(per_window_mse(persistence_forecast(TEST['context_state'], CFG.horizon), TEST['target_state']))
TEST_DYNAMICS = {}
for seed, summary in DYNAMICS.items():
    if summary['status'] == 'DONE':
        pred = predict_dataset(load_dynamics(seed), DATASETS['test'], CFG.eval_batch_size, DEVICE)
        TEST_DYNAMICS[seed] = {'model_rmse': horizon_mean_rmse(per_window_mse(pred['state_mean'], pred['target_state'])),
                               'persistence_rmse': persistence_test}
atomic_write_json(PATHS['artifacts'] / 'dynamics_test.json', TEST_DYNAMICS)

TEST_RESULTS = []


def record_test(name, function):
    try:
        function()
        TEST_RESULTS.append({'name': name, 'status': 'PASS'})
    except Exception as error:  # noqa: BLE001
        TEST_RESULTS.append({'name': name, 'status': 'FAIL', 'error': repr(error), 'traceback': traceback.format_exc()})


first_seed = next((s for s in CFG.seeds if DYNAMICS[s]['status'] == 'DONE'), None)
batch = move_batch(torch.utils.data.default_collate([DATASETS['test'][i] for i in range(CFG.batch_size)]), DEVICE)


def test_predictions_ignore_future_targets():
    model = load_dynamics(first_seed)
    tampered = dict(batch, target_state=torch.randn_like(batch['target_state']), target_visual=torch.randn_like(batch['target_visual']))
    with torch.no_grad():
        a, b = model.predict(batch), model.predict(tampered)
    assert all(torch.equal(a[k], b[k]) for k in a)


def test_hold_gain_is_exactly_zero():
    traces = ALL_TRACES
    assert len(traces) and (traces['gain_0'] == 0).all()


def test_checkpoint_roundtrip():
    model = load_dynamics(first_seed)
    clean = build_dynamics_model()
    clean.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()}, strict=True)
    assert all(torch.equal(v.cpu(), clean.state_dict()[k]) for k, v in model.state_dict().items())


def test_data_contract():
    assert DATA_AUDIT['status'] == 'PASS' and not DATA_AUDIT['synthetic_fallback_used']


record_test('data_contract', test_data_contract)
record_test('predictions_ignore_future_targets', test_predictions_ignore_future_targets)
record_test('hold_gain_is_exactly_zero', test_hold_gain_is_exactly_zero)
record_test('checkpoint_roundtrip', test_checkpoint_roundtrip)
atomic_write_json(PATHS['tests'] / 'test_results.json', TEST_RESULTS)
print(pd.DataFrame(TEST_RESULTS)[['name', 'status']].to_string())

# Latency of each deployable scorer at the same batch (exact co-state needs a backward pass and targets).
SYSTEMS = {'gpu': GPU_NAME, 'batch': CFG.batch_size}
if first_seed is not None and ALLOCATOR.get((first_seed, 'gate'), {}).get('status') == 'DONE':
    teacher = load_dynamics(first_seed)
    job = allocator_job(first_seed, 'gate', teacher).eval()
    targets = exact_targets(teacher, batch, CFG.candidate_cost_weight)
    latent, effects, costs = targets['latent'], targets['effects'], targets['costs']
    budget, horizon = job._conditions(latent)
    with torch.no_grad():
        SYSTEMS['critic_ms'] = measure_latency_ms(lambda: job.critic(latent, effects[:, 1:], costs[1:], budget, horizon), DEVICE)
        SYSTEMS['amortised_costate_ms'] = measure_latency_ms(lambda: job.costate(latent, budget, horizon), DEVICE)
    SYSTEMS['exact_costate_and_gains_ms'] = measure_latency_ms(lambda: exact_targets(teacher, batch, CFG.candidate_cost_weight), DEVICE, warmup=3, iterations=20)
    SYSTEMS['note'] = 'uncertainty-only needs one rollout per option; its cost is inside exact_costate_and_gains_ms here'
atomic_write_json(PATHS['benchmarks'] / 'systems_benchmark.json', SYSTEMS)

verdicts = [r['verdict'] for r in SEED_RESULTS.values() if r['status'] == 'EVALUATED']
REPORT = {
    'run_id': RUN_ID,
    'status': {
        'data': DATA_AUDIT['status'],
        'dynamics_gate_validation': {s: g['passed'] for s, g in DYNAMICS_GATE.items()},
        'tests': 'PASS' if all(t['status'] == 'PASS' for t in TEST_RESULTS) else 'FAIL',
        'seeds_evaluated': len(verdicts), 'seeds_planned': len(CFG.seeds),
    },
    'config_hash': CONFIG_HASH, 'data_manifest_hash': DATA_MANIFEST_HASH, 'source_hash': SOURCE_HASH,
    'split_parity_with_pilot': SPLIT_PARITY,
    'per_seed': SEED_RESULTS,
    'pooled_test': POOLED,
    'dynamics_test': TEST_DYNAMICS,
    'systems': SYSTEMS,
    'task_success': 'NOT_MEASURED (offline dataset; no simulator or robot)',
    'claim_boundary': ('Track E evidence on DROID-100 refinement candidates with a latent-state co-state. '
                       'Not a confirmatory H2 verdict (that needs N1/N2 and gate G-H2).'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# Pilot v2 `{RUN_ID}`', '', '| Seed | Opportunity (val) | Critic floor (val) | adjoint − critic (test) | 95 % CI | Verdict |', '|---|---|---|---:|---|---|']
for seed, r in SEED_RESULTS.items():
    if r['status'] != 'EVALUATED':
        lines.append(f"| {seed} | — | — | — | — | {r['status']} |")
        continue
    p = r['adjoint_minus_critic_test']
    lines.append(f"| {seed} | {r['opportunity_gate_validation']} | {r['critic_floor_validation']['passed']} | {p['estimate']:+.4f} | "
                 f"[{p['ci_low']:+.4f}, {p['ci_high']:+.4f}] | {r['verdict']} |")
lines += ['', 'Negative = the adjoint allocator has lower regret than the critic.', '', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
sync_logs()
if REPORT['status']['tests'] == 'PASS':
    atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

| Observation | Meaning | Next step |
|---|---|---|
| `NON_DIAGNOSTIC` | Per-window choice barely beats the best fixed option on validation, so no allocator can show an advantage | Redesign the candidates (roadmap §6), not the allocator; try the declared `single_choice_wta` variant as a new run |
| `UNDER_REALIZED` | The critic did not beat expected-random and uncertainty-only | Bounded direct-critic rescue (comprehensive plan §5.5.1) before any H2 statement |
| `TIE_OR_INCONCLUSIVE` | No detectable adjoint–critic difference | Report it as a tie; don't reword it |
| `ADJOINT_LOWER_REGRET` with `randomized − adjoint` CI > 0 | The co-state's alignment with ∂J/∂z carries the effect on this benchmark | Still Track E evidence only; confirm on N1/N2 |
| `ADJOINT_LOWER_REGRET` but the randomized control is as good | The effect is not specific to the co-state | Mechanism claim not allowed |
| `exact_costate` far better than `adjoint` | The first-order mechanism works; the estimator is weak | Improve the co-state estimator |

Import with the `import-run` skill (all JSON/CSV/parquet under `config/`, `tests/`, `artifacts/`, `benchmarks/`, `reports/`, `figures/`), then write the note with `research-note` and run `claim-check`.